# Solving the systems 

In [1]:
!pwd

/media/sf_vm_shared/Dev/ML-DSA/Security/Decompose/artifacts/artifacts/clean_template


In [2]:
# Loading auxiliary functions 
probable_path_to_helpers_functions = !find ../Common_functions -name "Helpers.py"
# If the Helper.py file is not found and you don't need it, comment this cell
# If the Helper.py file is not found and you need it, something went wrong ...
print(f"Probable path to Helpers functions:")
print(f">>> {probable_path_to_helpers_functions}")
probable_path_to_helpers_functions = probable_path_to_helpers_functions[0]
%run -i $probable_path_to_helpers_functions

Probable path to Helpers functions:
>>> ['../Common_functions/Helpers.py']


In [99]:
MODE = 2
# K = 3
# K = 5

In [100]:
# Loading ml-dsa parameters according to the chosen security level K
%run -i ../Common_functions/MLDSA_parameters.py {MODE}
# Loading auxiliary functions
%run -i ../Common_functions/MLDSA_functions.py
# Loading auxiliary functions
%run -i ../Common_functions/Additional_functions.py

In [101]:
# Importing useful libraries
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from tqdm.notebook import trange
import copy

import scared
import estraces

from collections import Counter

from scalib.preprocessing import Quantizer
from scalib.metrics import Ttest, SNR

import struct 

from scipy.signal import correlate
from scipy.ndimage import shift

import numpy as np

In [102]:
# Setting default size of figures in Matplotlib
plt.rcParams["figure.figsize"] = (13,3) 

# Set the maximum display width for NumPy arrays
np.set_printoptions(linewidth=sys.maxsize)

# Adjusting the display of tables
np.set_printoptions(threshold=sys.maxsize)

In [103]:
# Matplotlib constants 
span_color = "#FFC069"
color0 = "darkblue"
color1 = "coral"

In [106]:
for nb_key in range(5):
    temp = np.load("./stats/Keys/Key"+str(nb_key)+"/w_shares.npz")
    w_share0 = temp['w_share0']
    w_share1 = temp['w_share1']
    
    temp2 = np.load("./stats/Keys/Key"+str(nb_key)+"/w_cs2_shares.npz")
    wmcs2_share0 = temp2['w_cs2_share0']
    wmcs2_share1 = temp2['w_cs2_share1']
    wmcs2 = [int((wmcs2_share0[_] + wmcs2_share1[_])%Q) for _ in range(len(wmcs2_share0))]
    print(len(wmcs2))
    w = [int((w_share0[_] + w_share1[_])%Q) for _ in range(len(w_share0))]

    K = 4
    N =256
    
    #Data
    flat_array = np.fromfile('./stats/Keys/Key'+str(nb_key)+'/A_data.bin', dtype=np.int8)
    A = flat_array.reshape(-1, 256)
    flat_array = np.fromfile('./stats/Keys/Key'+str(nb_key)+'/s2_data.bin', dtype=np.int32)
    s2 = flat_array.reshape(K, N)

    temp = np.load("./stats/Keys/Key"+str(nb_key)+"/RES.npz")
    b = list(temp["val_g"])
    print(len(b))
    indices_lignes = list(temp["ind_w"])
    A = A[indices_lignes,:]
    nb_eq = indices_lignes[-1]
    wmcs2 = [wmcs2[_] for _ in range(len(wmcs2)) if _ in indices_lignes]
    print(nb_eq,len(b),len(wmcs2))
    print("shape A",np.shape(A))
    corr_b = [b[_] - wmcs2[_] for _ in range(len(b))]

    x, _, _, _ = np.linalg.lstsq(A, corr_b)
    print("nb_key: ",nb_key)
    print("key?",np.array_equal(np.round(x).astype(int), s2[0]))
    print("Nb eq:",indices_lignes[-1])
    print("Wrong coeffs:",np.count_nonzero(np.round(x).astype(int) - s2[0]))
    print("---")

12728

In [ ]:
## Statistics:

In [ ]:
stair_center = 0
usable_sc = 0 

for nb_key in range(5):
    temp = np.load("./stats/5_keys/Key"+str(nb_key)+"/w_shares.npz")
    w_share0 = temp['w_share0']
    w_share1 = temp['w_share1']
    
    temp2 = np.load("./stats/5_keys/Key"+str(nb_key)+"/w_cs2_shares.npz")
    wmcs2_share0 = temp2['w_cs2_share0']
    wmcs2_share1 = temp2['w_cs2_share1']
    wmcs2 = [int((wmcs2_share0[_] + wmcs2_share1[_])%Q) for _ in range(len(wmcs2_share0))]
    #print(len(wmcs2))
    w = [int((w_share0[_] + w_share1[_])%Q) for _ in range(len(w_share0))]

    K = 4
    N =256
    
    #Data
    flat_array = np.fromfile('./stats/5_keys/Key'+str(nb_key)+'/A_data.bin', dtype=np.int8)
    A = flat_array.reshape(-1, 256)
    flat_array = np.fromfile('./stats/5_keys/Key'+str(nb_key)+'/s2_data.bin', dtype=np.int32)
    s2 = flat_array.reshape(K, N)

    temp = np.load("./stats/5_keys/Key"+str(nb_key)+"/RES.npz")
    b = list(temp["val_g"])
    #print(len(b))
    indices_lignes = list(temp["ind_w"])
    A = A[indices_lignes,:]
    nb_eq = indices_lignes[-1]
    wmcs2 = [wmcs2[_] for _ in range(len(wmcs2)) if _ in indices_lignes]
    #print(nb_eq,len(b),len(wmcs2))
    #print("shape A",np.shape(A))
    corr_b = [b[_] - wmcs2[_] for _ in range(len(b))]

    x, _, _, _ = np.linalg.lstsq(A, corr_b)
    print("nb_key: ",nb_key)
    print("Collected stair points:",len(w))
    print("Usable stair points", len(b))
    print("key?",np.array_equal(np.round(x).astype(int), s2[0]))
    stair_center +=len(w)
    usable_sc+= len(b)
    print("---")

In [ ]:
print("Average number of stair center",stair_center/5)
print("Average number of used stair center", usable_sc/5)

In [ ]:
eq = 0

for nb_key in range(5):
    temp = np.load("./stats/5_keys/Key"+str(nb_key)+"/w_shares.npz")
    w_share0 = temp['w_share0']
    w_share1 = temp['w_share1']
    
    temp2 = np.load("./stats/5_keys/Key"+str(nb_key)+"/w_cs2_shares.npz")
    wmcs2_share0 = temp2['w_cs2_share0']
    wmcs2_share1 = temp2['w_cs2_share1']
    wmcs2 = [int((wmcs2_share0[_] + wmcs2_share1[_])%Q) for _ in range(len(wmcs2_share0))]
    w = [int((w_share0[_] + w_share1[_])%Q) for _ in range(len(w_share0))]

    K = 4
    N =256
    
    #Data
    flat_array = np.fromfile('./stats/5_keys/Key'+str(nb_key)+'/A_data.bin', dtype=np.int8)
    A = flat_array.reshape(-1, 256)
    flat_array = np.fromfile('./stats/5_keys/Key'+str(nb_key)+'/s2_data.bin', dtype=np.int32)
    s2 = flat_array.reshape(K, N)

    temp = np.load("./stats/5_keys/Key"+str(nb_key)+"/RES.npz")
    b = list(temp["val_g"])
    #print(len(b))
    indices_lignes = list(temp["ind_w"])
    A = A[indices_lignes,:]
    nb_eq = indices_lignes[-1]
    wmcs2 = [wmcs2[_] for _ in range(len(wmcs2)) if _ in indices_lignes]
    w = [w[_] for _ in range(len(w)) if _ in indices_lignes]
    corr_b = [b[_] - wmcs2[_] for _ in range(len(b))]

    x, _, _, _ = np.linalg.lstsq(A, corr_b)
    print("nb_key: ",nb_key)
    print("Number of correct equations:", len(w) - np.count_nonzero(np.array(w) - np.array(b)))
    eq += len(w) - np.count_nonzero(np.array(w) - np.array(b))
    print("---")

In [ ]:
print("Average number of correct equations",eq/5)